# 1907. Count Salary Categories

## Problem Description

A bank stores each account's monthly income and wants a distribution across three salary groups.

### Table Schema

**Accounts**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| account_id | int | unique account identifier (key) |
| income | int | income amount |

### Task

Classify each account into Low Salary (income < 20000), Average Salary (20000 ≤ income ≤ 50000), or High Salary (income > 50000). Return category and accounts_count with exactly one row for each of the three labels, even when a group has zero accounts. Any row order is accepted.

## Example

### Input

**accounts**

| account_id | income |
| --- | --- |
| 1 | 19999 |
| 2 | 20000 |
| 3 | 50000 |
| 4 | 50001 |
| 5 | 80000 |

### Expected Output

| category | accounts_count |
| --- | --- |
| Low Salary | 1 |
| Average Salary | 2 |
| High Salary | 2 |

### Explanation

19999 is Low; 20000 and 50000 are Average; 50001 and 80000 are High. The counts are 1, 2, and 2.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1900-1999/1907.Count%20Salary%20Categories/README_EN.md) · [LeetCode](https://leetcode.com/problems/count-salary-categories/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/count-salary-categories/solutions/) · [Individual community solution](https://leetcode.cn/problems/count-salary-categories/solutions/2375460/1907-an-fen-lei-tong-ji-xin-shui-by-yyz9-iwrf/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
from decimal import Decimal, ROUND_HALF_UP

def percent(n,d):
    return float((Decimal(int(n))*100/Decimal(int(d))).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)) if d else None

accounts = pd.DataFrame([[1,19999],[2,20000],[3,50000],[4,50001],[5,80000]], columns=["account_id","income"])
expected = pd.DataFrame([["Low Salary",1],["Average Salary",2],["High Salary",2]], columns=["category","accounts_count"])
display(accounts)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Boolean counts
邊界含 20000、50000。

In [ ]:
def solution_1(accounts):
    s = accounts.income
    return pd.DataFrame({'category':['Low Salary','Average Salary','High Salary'],'accounts_count':[s.lt(20000).sum(),s.between(20000,50000).sum(),s.gt(50000).sum()]})

result_1 = solution_1(accounts.copy())
display(result_1)

### Solution 2 — Cut and reindex
income 為整數；cut 可處理 bins。

In [ ]:
def solution_2(accounts):
    labels = ['Low Salary','Average Salary','High Salary']
    c = pd.cut(accounts.income,[-float('inf'),19999,50000,float('inf')],labels=labels)
    return c.value_counts().reindex(labels,fill_value=0).rename_axis('category').reset_index(name='accounts_count')

result_2 = solution_2(accounts.copy())
display(result_2)

### Solution 3 — Mask and value counts
mask 條件賦值；reindex 補零。

In [ ]:
def solution_3(accounts):
    s = accounts.income
    labels = ['Low Salary','Average Salary','High Salary']
    c = pd.Series('Average Salary',index=s.index).mask(s.lt(20000),labels[0]).mask(s.gt(50000),labels[2])
    return c.value_counts().reindex(labels,fill_value=0).rename_axis('category').reset_index(name='accounts_count')

result_3 = solution_3(accounts.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare readability, intermediate data size and handling of duplicate values, missing matches and empty groups.